# DV3 — Verwerking van advieselementen door kabinetsreacties

**Run van 5 juni 2026.** Dit notebook beantwoordt deelvraag 3 (DV3): *hoe verhouden kabinetsreacties zich tot afzonderlijke probleemdefinities en aanbevelingen, en verschilt die verwerking tussen beide elementtypen?*

Alle cijfers worden **live en re-runbaar** uit de PostgreSQL-database berekend via `dv3_analyse.py` (geen hardcoded aantallen; klopt ook als er later data bijkomt). De labels zijn op individueel elementniveau zwak gemeten (zie DV2, Krippendorff α ≈ 0,19); DV3 rapporteert daarom uitsluitend **geaggregeerde aandelen**, met meetkwaliteit als gevoeligheid.

**Changelog**
- 2026-06-04: Eerste versie. Noemer = match-rijen, DV2-gefilterd via het advies-document. Twee dimensies uit `finale_verwerkingsitem.match_bewijs`.
- 2026-06-05: Scope gelijkgetrokken naar de 171-Kaderwet-scope (`scope_fasen`-CTE), zoals DV1/DV2, in plaats van `dashboard_visible`. Noemer hierdoor 15.368 elementen / 482 reacties / 400 adviezen. Cijfers bevroren in `DV2_validatie_bronnen/11_dv3_verwerking_171_frozen_20260605/`.

**Input:** `pipeline.kabinetsreactie_aanbeveling_matches`, `pipeline.kabinetsreactie_analyse`, `corpus.adviesdocumenten`, `register.adviescollege_fasen`.  
**Output:** tabellen voor de DV3-resultatensectie van de scriptie.

## Methode: noemer, filter en 11→6 klassenmapping

De scope-filter is de **171-Kaderwet-scope** (`scope_fasen`-CTE: `kaderwet IS TRUE` + een geldig `instellingsbesluit_document_id` + `tijd_adviescollege` van het type Permanent/Tijdelijk/Eenmalig, met uitsluiting van Autoriteit Persoonsgegevens en College Bescherming Persoonsgegevens), gelijk aan DV1/DV2. Dit vervangt de oudere `dashboard_visible`-afbakening. De filter wordt toegepast **via het advies-document** (`a.advies_document_id`), niet via de reactie — reacties zijn Kamerstukken zonder college-link en zouden anders onterecht wegvallen. Daarnaast tellen alleen post-fix verwerkingsruns mee (`r.started_at >= '2026-06-03'`). De 11 labels worden samengevat tot 6 grove klassen volgens de DV2-confusion-matrix.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import dv3_analyse as dv3

print('171-Kaderwet-scope filter (via advies-document):')
print(dv3.FILTER_SQL)
print('11->6 klassenmapping:')
for k, v in dv3.KLASSE6.items():
    print(f'  {k:32s} -> {v}')

## Stap 1 — Laden, noemer en dekking

Eén rij per advies-element × reactie. De dekkingscontrole laat zien hoeveel reacties wél/geen elementen opleverden.

In [ ]:
res = dv3.run()
df = res['_df']
print('NOEMER (171-Kaderwet-scope):', res['n_elementen'], 'elementen /',
      res['n_reacties'], 'reacties /', res['n_adviezen'], 'adviezen')
print('Per elementtype:', res['per_type_n'])
print('Dekking (alle reacties):', res['dekking'])
print('Elementen per reactie:', res['elementen_per_reactie'])
print('\nReferentie: bevroren bundel 11 (peildatum 2026-06-05) noemt'
      ' 15.368 / 482 / 400; bovenstaande is de actuele DB-stand op de 171-scope.')

## Stap 3 — Corpusbrede verdeling (11 labels + 6 klassen)

Met 95%-Wilson-betrouwbaarheidsintervallen op de aandelen.

In [ ]:
from IPython.display import display
print('6 grove klassen:'); display(res['verdeling_6'])
print('11 labels:'); display(res['verdeling_11'])

## Stap 4 — Verdeling per elementtype

De kern van DV3: verschilt de verwerking tussen aanbevelingen en probleemdefinities?

In [ ]:
for et in ['aanbeveling', 'probleemdefinitie']:
    print(f'6-klasse | {et}'); display(res['verdeling_6_per_type'][et])
for et in ['aanbeveling', 'probleemdefinitie']:
    print(f'11-label | {et}'); display(res['verdeling_11_per_type'][et])

## Stap 5 — Twee onderliggende dimensies per elementtype

Het label volgt uit (kabinetspositie × beleidsmatige opvolging). Deze waarden komen uit `finale_verwerkingsitem.match_bewijs` (modale waarde per element). Bij `onduidelijk`/`niet_herkenbaar_verwerkt` is er geen positie → `onbekend`; dat verklaart het hoge `onbekend`-aandeel, vooral bij probleemdefinities.

In [ ]:
print('Kabinetspositie:')
for et in ['aanbeveling', 'probleemdefinitie']:
    print(f'  [{et}]'); display(res['dim_positie_per_type'][et])
print('Beleidsmatige opvolging:')
for et in ['aanbeveling', 'probleemdefinitie']:
    print(f'  [{et}]'); display(res['dim_opvolging_per_type'][et])

## Stap 6 — Statistische vergelijking elementtype × verwerking

Cramér's V als hoofdmaat (bij n≈11.000 is vrijwel alles 'significant'). **Clustering-voorbehoud:** elementen binnen één reactie zijn niet onafhankelijk, dus de chi²-p-waarde is te liberaal. De clustered-spreiding herberekent V op 100 herbemonsteringen van 1 willekeurig element per reactie (seed=100).

In [ ]:
print('11-label  V =', res['cramers_11']['cramers_v'],
      '| clustered mediaan/p05/p95 =', res['cramers_11_clustered'])
print('6-klasse  V =', res['cramers_6']['cramers_v'],
      '| clustered mediaan/p05/p95 =', res['cramers_6_clustered'])
print('chi2 (11-label) =', res['cramers_11']['chi2'], 'p =', res['cramers_11']['p_value'])
print('Elementen per reactie (clustering-context):', res['elementen_per_reactie'])

## Stap 7 — Gevoeligheid 'onduidelijk' (3 noemer-varianten)

(1) inclusief alles, (2) exclusief `onduidelijk`, (3) exclusief beide vangnetcategorieën. Toont hoe sterk de aandelen van die onzekere categorie afhangen.

In [ ]:
display(res['sensitivity'])

## Stap 8 — Meetkwaliteit-stratificatie (proxy)

Transparante, grove proxy: reacties gegroepeerd naar hun aandeel `onduidelijk` (laag <33% / midden / hoog ≥67%). Dit is **geen** exacte heruitvoering van het DV2-usability-kader, maar maakt zichtbaar dat substantiële verwerking samenhangt met de meetkwaliteit van een reactie.

In [ ]:
display(res['meetkwaliteit'])

## Stap 9 — Context en conclusie

Elementen per advies puur als context; **geen** vergelijking tussen colleges als zelfstandige bevinding (meetkwaliteit verschilt per college, zie DV2). DV3 is corpusbreed.

**Kernbevinding:** aanbevelingen en probleemdefinities worden aantoonbaar verschillend behandeld (Cramér's V ≈ 0,44 op 11 labels; robuust onder clustering). Het verschil zit vooral in het hoge aandeel `onduidelijk`/`niet_herkenbaar_verwerkt` bij probleemdefinities — een meet- én verwerkingskenmerk dat binnen de DV2-betrouwbaarheidsgrenzen moet worden gelezen.

In [ ]:
print('Elementen per advies:', res['elementen_advies'] if 'elementen_advies' in res
      else res['elementen_per_advies'])
print('Aantal colleges in scope:', df['college'].nunique())